# Loading Yahoo Fantasy Data into Iceberg

This notebook demonstrates how to:
1. Authenticate with Yahoo Fantasy API
2. Run the pipeline to fetch and transform data
3. Persist data to Iceberg tables
4. Query the loaded data

## Setup: Imports and Configuration

In [1]:
import json
import os
from pathlib import Path

import polars as pl
from pyiceberg.catalog import load_catalog

from nfl.yahoo_fantasy.auth import build_oauth_session, load_token
from nfl.yahoo_fantasy.pipeline import run_pipeline, PipelineConfig
from nfl.yahoo_fantasy.storage.iceberg import IcebergCatalogConfig, IcebergNamespaceConfig, WriteMode


In [2]:
# Configure Polars display
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

## Step 1: Locate Project Root and Load Catalog

In [2]:
cwd = Path.cwd()

# Resolve repository root from cwd by walking up to pyproject.toml.
def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    while current != current.parent:
        if (current / "pyproject.toml").exists():
            if current.name.lower() == "examples":
                raise RuntimeError("Invalid project root resolved to 'examples'.")
            return current
        current = current.parent
    raise RuntimeError("Cannot locate project root. Expected pyproject.toml in repository root.")

project_root = find_project_root(cwd)

# Use repo root as cwd so PyIceberg can use relative local paths reliably on Windows.
os.chdir(project_root)

CATALOG_URI = "sqlite:///iceberg_catalog.db"
WAREHOUSE = "./warehouse"

catalog = load_catalog(
    "yahoo",
    type="sql",
    uri=CATALOG_URI,
    warehouse=WAREHOUSE,
)

print("✓ Catalog loaded.")
print("Working directory:", Path.cwd())
print("Catalog URI:", CATALOG_URI)
print("Warehouse:", WAREHOUSE)

✓ Catalog loaded.
Working directory: C:\Users\EricTruett\nfl
Catalog URI: sqlite:///iceberg_catalog.db
Warehouse: ./warehouse


## Step 2: Set Up OAuth Authentication

### Getting Yahoo Developer Credentials

You need real credentials registered with Yahoo:

1. **Go to [Yahoo Developer Network](https://developer.yahoo.com/apps)**
2. **Create a new app** with:
   - App Name: (e.g., "NFL Fantasy Data Tool")
   - Application Type: **OAuth 2.0**
   - Redirect URI(s): `http://localhost:8000`
3. **Copy your credentials:**
   - Client ID (consumer key)
   - Client Secret (consumer secret)

⚠️ **Important:** Do NOT use placeholder values like `YOUR_YAHOO_CLIENT_ID`. The error "Invalid consumer key" means you're using a value that doesn't exist in Yahoo's system.

### Store Credentials

Create `.secrets/credentials.json` in the project root with your **actual** credentials:

```json
{
  "client_id": "your_actual_client_id_from_yahoo",
  "client_secret": "your_actual_client_secret_from_yahoo",
  "redirect_uri": "http://localhost:8000"
}
```

On first run, you'll be guided through the OAuth flow to authorize the app. The token will be cached in `.secrets/yahoo_token.json`.

In [3]:
# Helper: Create .secrets/credentials.json if needed
secrets_dir = project_root / ".secrets"
secrets_file = secrets_dir / "credentials.json"

if not secrets_file.exists():
    print("Creating .secrets directory and credentials template...")
    secrets_dir.mkdir(exist_ok=True)
    
    template = {
        "client_id": "YOUR_YAHOO_CLIENT_ID",
        "client_secret": "YOUR_YAHOO_CLIENT_SECRET",
        "redirect_uri": "http://localhost:8000"
    }
    
    with open(secrets_file, "w") as f:
        json.dump(template, f, indent=2)
    
    print(f"✓ Created {secrets_file}")
    print("  Edit this file with your actual credentials from Yahoo Developer Network")
else:
    print(f"✓ Credentials file already exists at {secrets_file}")

✓ Credentials file already exists at C:\Users\EricTruett\nfl\.secrets\credentials.json


In [4]:
# Get OAuth credentials from environment or use cached token
token_path = project_root / ".secrets" / "yahoo_token.json"
cached_token = load_token(token_path)

# If token exists, use it; otherwise get credentials from environment
if cached_token:
    # Use cached token with minimal setup
    client_id = os.getenv("YAHOO_CLIENT_ID", "")
    client_secret = os.getenv("YAHOO_CLIENT_SECRET", "")
else:
    client_id = os.getenv("YAHOO_CLIENT_ID")
    client_secret = os.getenv("YAHOO_CLIENT_SECRET")
    if not client_id or not client_secret:
        raise ValueError(
            "OAuth credentials required. Set YAHOO_CLIENT_ID and YAHOO_CLIENT_SECRET environment variables."
        )

redirect_uri = os.getenv("YAHOO_REDIRECT_URI", "http://localhost:8000")

oauth_session = build_oauth_session(
    client_id=client_id,
    client_secret=client_secret,
    redirect_uri=redirect_uri,
    token_path=token_path,
    auth_code=os.getenv("YAHOO_AUTH_CODE", ""),
    open_browser=False,
)

print("✓ OAuth session established.")

✓ OAuth session established.


## Step 3: Configure and Run the Pipeline

In [5]:
# Configure Iceberg settings
iceberg_config = IcebergCatalogConfig(
    uri=CATALOG_URI,
    warehouse=WAREHOUSE,
)

# Configure the pipeline
config = PipelineConfig(
    storage_target="iceberg",              # Save to Iceberg tables
    iceberg_catalog=iceberg_config,         # Use our catalog
    iceberg_mode="upsert",                 # Upsert mode (idempotent writes)
    iceberg_dry_run=False,                  # Set to True to test without writing
    validate_contracts=True,                # Validate data contracts
    materialized_views_enabled=False,       # Optional: enable for derived tables
    standardization_enabled=False,          # Optional: enable for entity standardization
)

print("Pipeline configuration ready.")
print(f"  Storage target: {config.storage_target}")
print(f"  Write mode: {config.iceberg_mode}")
print(f"  Dry run: {config.iceberg_dry_run}")

Pipeline configuration ready.
  Storage target: iceberg
  Write mode: upsert
  Dry run: False


## Step 4: Fetch Your League Key and Run Pipeline

You can find your league key by:
- Visiting your Yahoo Fantasy league page and copying it from the URL (format: `###.l.#####`)
- Or programmatically fetching it from the API

In [12]:
# Option 1: Specify your league key directly
LEAGUE_KEY = "461.l.717896"  # CHANGE THIS to your actual league key
SPORT = "nfl"  # or "nba"

print(f"Running pipeline for league: {LEAGUE_KEY}")
print(f"Sport: {SPORT}")

result = run_pipeline(
    league_key=LEAGUE_KEY,
    sport=SPORT,
    oauth_session=oauth_session,
    config=config,
)

print("\n✓ Pipeline completed!")
print(f"  Frames extracted: {list(result.frames.keys())}")
print(f"  Iceberg outputs: {len(result.iceberg_outputs)} tables written")
if result.iceberg_outputs:
    for output in result.iceberg_outputs:
        print(f"    - {output.table_identifier}: {output.written_rows} rows")

Running pipeline for league: 461.l.717896
Sport: nfl

✓ Pipeline completed!
  Frames extracted: ['league', 'team', 'player', 'draft_pick', 'transaction', 'stat_category', 'scoring_rule', 'nfl_standings', 'nfl_matchups', 'nfl_roster_entries', 'nfl_player_stats_weekly', 'nba_standings', 'nba_standing_category_scores', 'nba_roster_entries', 'nba_player_projections']
  Iceberg outputs: 15 tables written
    - yahoo_common.league: 1 rows
    - yahoo_common.team: 12 rows
    - yahoo_common.player: 1187 rows
    - yahoo_common.draft_pick: 168 rows
    - yahoo_common.transaction: 624 rows
    - yahoo_common.stat_category: 86 rows
    - yahoo_common.scoring_rule: 30 rows
    - yhnfl.standings: 12 rows
    - yhnfl.matchups: 0 rows
    - yhnfl.roster_entries: 1841 rows
    - yhnfl.player_stats_weekly: 1841 rows
    - ynbna.standings: 0 rows
    - ynbna.standing_category_scores: 0 rows
    - ynbna.roster_entries: 0 rows
    - ynbna.player_projections: 0 rows


## Step 5: List Available Tables

In [13]:
def list_tables(namespace: str) -> list[str]:
    try:
        return [f"{ns}.{tbl}" for ns, tbl in catalog.list_tables(namespace)]
    except Exception as exc:
        print(f"Could not list tables for namespace '{namespace}': {exc}")
        return []

for ns in ["yahoo_common", "yhnfl"]:
    tables = list_tables(ns)
    print(f"\n{ns}:")
    if tables:
        for t in tables:
            print(f"  ✓ {t}")
    else:
        print("  (no tables found)")


yahoo_common:
  ✓ yahoo_common.draft_pick
  ✓ yahoo_common.league
  ✓ yahoo_common.player
  ✓ yahoo_common.scoring_rule
  ✓ yahoo_common.stat_category
  ✓ yahoo_common.team
  ✓ yahoo_common.transaction

yhnfl:
  ✓ yhnfl.player_stats_weekly
  ✓ yhnfl.roster_entries
  ✓ yhnfl.standings


## Step 6: Load and Query Data

In [14]:
def load_table_as_polars(table_identifier: str) -> pl.DataFrame:
    """Load an Iceberg table into a Polars DataFrame."""
    table = catalog.load_table(table_identifier)
    arrow_table = table.scan().to_arrow()
    return pl.from_arrow(arrow_table)

def maybe_load(table_identifier: str) -> pl.DataFrame | None:
    try:
        return load_table_as_polars(table_identifier)
    except Exception as exc:
        print(f"Skipping '{table_identifier}': {exc}")
        return None

### Example: Load League and Team Data

In [15]:
league_df = maybe_load("yahoo_common.league")
team_df = maybe_load("yahoo_common.team")

if league_df is not None:
    print("League data:")
    print(league_df.head())
    
if team_df is not None:
    print("\nTeam data:")
    print(team_df.head())

League data:
shape: (1, 9)
┌────────────┬───────────┬─────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ league_key ┆ league_id ┆ game_id ┆ game_code ┆ … ┆ league_na ┆ scoring_t ┆ league_ty ┆ num_teams │
│ ---        ┆ ---       ┆ ---     ┆ ---       ┆   ┆ me        ┆ ype       ┆ pe        ┆ ---       │
│ str        ┆ str       ┆ i64     ┆ str       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ i64       │
│            ┆           ┆         ┆           ┆   ┆ str       ┆ str       ┆ str       ┆           │
╞════════════╪═══════════╪═════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 461.l.7178 ┆ 717896    ┆ 461     ┆ nfl       ┆ … ┆ Dilly     ┆ head      ┆ private   ┆ 12        │
│ 96         ┆           ┆         ┆           ┆   ┆ Dilly     ┆           ┆           ┆           │
└────────────┴───────────┴─────────┴───────────┴───┴───────────┴───────────┴───────────┴───────────┘

Team data:
shape: (5, 6)
┌───────────────────┬──────────────┬──

### Example: Load and Summarize Standings

In [16]:
standings_df = maybe_load("yhnfl.standings")

if standings_df is not None:
    standings_summary = (
        standings_df
        .sort(["season", "wins", "points_for"], descending=[False, True, True])
        .select([
            "league_key",
            "season",
            "team_key",
            "rank",
            "wins",
            "losses",
            "ties",
            "points_for",
            "points_against",
        ])
    )
    print("Standings summary:")
    print(standings_summary)

Standings summary:
shape: (12, 9)
┌──────────────┬────────┬─────────────────┬──────┬───┬────────┬──────┬────────────┬────────────────┐
│ league_key   ┆ season ┆ team_key        ┆ rank ┆ … ┆ losses ┆ ties ┆ points_for ┆ points_against │
│ ---          ┆ ---    ┆ ---             ┆ ---  ┆   ┆ ---    ┆ ---  ┆ ---        ┆ ---            │
│ str          ┆ i64    ┆ str             ┆ i64  ┆   ┆ i64    ┆ i64  ┆ f64        ┆ f64            │
╞══════════════╪════════╪═════════════════╪══════╪═══╪════════╪══════╪════════════╪════════════════╡
│ 461.l.717896 ┆ 2025   ┆ 461.l.717896.t. ┆ 4    ┆ … ┆ 3      ┆ 0    ┆ 1854.52    ┆ 1540.48        │
│              ┆        ┆ 6               ┆      ┆   ┆        ┆      ┆            ┆                │
│ 461.l.717896 ┆ 2025   ┆ 461.l.717896.t. ┆ 1    ┆ … ┆ 4      ┆ 0    ┆ 1838.4     ┆ 1672.76        │
│              ┆        ┆ 11              ┆      ┆   ┆        ┆      ┆            ┆                │
│ 461.l.717896 ┆ 2025   ┆ 461.l.717896.t. ┆ 3    ┆ … ┆ 5 

## Troubleshooting

### No tables found after running pipeline
- Check that `iceberg_dry_run` is set to `False`
- Verify the warehouse directory exists at `{project_root}/warehouse`
- Check the pipeline result for any errors

### OAuth authentication fails
- Ensure your Yahoo Developer credentials are configured
- Check that tokens are stored correctly (usually in `.cache` or environment variables)
- See [YAHOO_INTEGRATION_BLUEPRINT.md](../blueprints/YAHOO_INTEGRATION_BLUEPRINT.md) for detailed setup

### Data contract validation errors
- Set `validate_contracts=False` to skip validation
- Check the API response matches expected schema
- See test files in `tests/` for contract definitions